# Fase 3 — perfiles sociodemográficos, modelos predictivos y NBI 2022

**Grupo 4 · Ciencia de Datos / EPHC 2024**

Pregunta: ¿Qué perfiles sociodemográficos y socioeconómicos presentan mayor prevalencia de enfermedad reciente y en qué medida sus características permiten clasificar esta condición?

**Importante:** la enfermedad es autorreportada en los últimos 90 días; no es predicción clínica futura. La información NBI es departamental y contextual.

## 1. Reproducibilidad y datos
El pipeline reutiliza la EPHC procesada y los cuatro archivos trimestrales de las fases anteriores. Mantiene la separación T1–T3 para desarrollo y T4 para prueba final.

In [ ]:
from src.fase3.comun import cargar, validar_division_temporal, SALIDA
from src.fase3 import perfiles, clasificacion, nbi
import pandas as pd
from IPython.display import display

df = cargar()
validar_division_temporal(df)
print('Personas:', len(df), '| Trimestres:', df['trimestre'].value_counts().sort_index().to_dict())

## 2. Segmentación de perfiles
Se construyen clusters con características demográficas y socioeconómicas. **No** se utiliza enfermedad reciente para formar los grupos. El k se elige por silhouette sobre una muestra del entrenamiento.

In [ ]:
r_perfiles = perfiles.ejecutar(df)
print('Selección de clusters:', r_perfiles['clusters'], 'silhouette:', round(r_perfiles['silhouette'], 4))
display(pd.read_csv(SALIDA / 'silhouette_k.csv'))
display(pd.read_csv(SALIDA / 'resumen_perfiles.csv'))

## 3. Clasificación supervisada
Regresión logística, bosque aleatorio y Naive Bayes frente a baseline. Validación temporal T1 → T2 y T1+T2 → T3. Se selecciona por AUC-ROC de validación; T4 se reserva hasta el final.

In [ ]:
r_clas = clasificacion.ejecutar(df)
print('Mejor modelo de validación:', r_clas['modelo'])
display(pd.read_csv(SALIDA / 'metricas_clasificacion.csv').round(3))
display(pd.read_csv(SALIDA / 'errores_subgrupos.csv').head(25).round(3))

## 4. Contexto territorial NBI
El NBI describe hogares del Censo 2022, mientras que la EPHC 2024 registra personas. El cruce ocurre únicamente **por departamento** y las correlaciones son exploratorias; no implican causalidad.

In [ ]:
r_nbi = nbi.ejecutar(df)
display(pd.read_csv(SALIDA / 'cruce_departamental_nbi.csv').round(2))
display(pd.read_csv(SALIDA / 'correlaciones_nbi.csv').round(3))

## 5. Interpretación y limitaciones
Completar con los resultados calculados: comparar con baseline, investigar errores y sobreajuste, explicar perfiles de mayor prevalencia y valorar la magnitud de las correlaciones. Una métrica desfavorable también es un resultado válido.

No usar estas probabilidades para decisiones médicas o de acceso a cobertura.